# 代理型人工智慧基礎與應用實作 | 期末專題
## 🤖 笨笨掃地機 — 以 LLM 為大腦、會「賣蠢」的 2.5D 掃地機器人指令遊戲

玩家扮演屋主，**只能用中文文字**指揮一台掃地機器人「笨笨」；笨笨的大腦是課程提供的 LLM。
LLM 只能從固定的**動作選單**中挑選並以 **JSON** 回覆，程式驗證白名單後交給遊戲引擎執行，
碰撞、闖禁區等**事件再回饋給 LLM**，形成完整的 Agent 閉環。

> 🎮 **v2 新玩法**（靈感：《教妹妹開車》——坐在副駕駛只能動口、對方不一定照做、語氣太兇還會鬧脾氣）：
> 笨笨有**心情**，你怎麼說話會影響它聽不聽話；它出包會讓**主人血壓**飆高；
> 還有電量、突發事件、分數連擊、三星評價、成就、關卡解鎖與每日挑戰。
>
> 🆕 **v3**：**10 個難度遞增的關卡**（笨笨會「恍神」不照做、記性差、導航壞掉、停電、步數限制……）、
> 像遊戲一樣的**遊戲視窗**（可以**彈出成獨立視窗**或全螢幕）、清楚的**方向標示**、對話框下方的**指令紀錄**，
> 以及**語音輸入**的可行性評估（`docs/voice_input.md`）。

| Notebook | 內容 |
|---|---|
| **01_快速開始_遊玩**（這本） | 連線 API → 看玩法 → 開始玩 |
| 02_Agent_原理展示 | 遊戲世界、Agent（Prompt／世界描述／記憶／JSON 驗證）、玩法系統、Session 閉環 |
| 03_評估 | 照做率、格式正確率、記憶遵守率、延遲、**心情 vs 照做率實驗**、遊戲 log 分析 |

程式碼都在專案根目錄的 `bendan/` 套件裡，Notebook 只負責「展示」與「遊玩」。

## 0｜載入套件

需要：`openai`、`ipywidgets`（遊戲視窗）、`pandas`／`matplotlib`（評估用）。

> ⚠️ **課程 JupyterHub 請選「Python 3 (ipykernel)」**（實測：它有全部套件和 API 設定；「Python (Lab 1)」沒有 ipywidgets／pandas，只能玩文字模式、不能跑評估）。
> 下面的檢查表會列出缺什麼。

In [ ]:
# 讓 Notebook 找得到 bendan 套件（不管從專案根目錄或 notebooks/ 執行都可以）
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "bendan").is_dir())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from IPython.display import HTML, Image, display
import bendan
print("✅ bendan", bendan.__version__, "｜專案位置：", ROOT)
bendan.check_env();   # 課程 JupyterHub 請用「Python 3 (ipykernel)」：它有 ipywidgets（遊戲視窗）、pandas、matplotlib

## 1｜連線課程 LLM API

和 API Quickstart 一樣使用 **OpenAI-compatible API**（課堂 vLLM）。
課堂 JupyterHub 已預先設定 `OPENAI_BASE_URL`、`OPENAI_API_KEY`，**不要把 Key 寫進程式或公開**。
找不到設定時會自動切換成**離線 FakeLLM**（規則式假模型），也能完整測試遊戲。

In [ ]:
from bendan import llm

llm.connect()          # 有 OPENAI_BASE_URL / OPENAI_API_KEY 就用課程 API，沒有就用離線 FakeLLM
llm.list_models();

### 1B. 選模型

提案預設使用 **Qwen2.5-7B**；也可以改成 `qwen2.5:3b`、`llama3.1:8b` 比較不同模型「賣蠢」的程度。

In [ ]:
MODEL = "qwen2.5:7b"   # 也可試 qwen2.5:3b 或 llama3.1:8b
if not llm.STATE.use_fake:
    llm.set_model(MODEL)
print("Selected model:", llm.model_name())

### 1C. 量一次延遲

延遲過長是提案中的風險之一；遊戲用**背景執行緒＋思考中動畫**把等待時間藏起來。

In [ ]:
r = llm.latency_test()
print(r["text"])
print(f"⏱️ 延遲 {r['latency']:.2f} 秒｜Token 用量：{r['usage']}")

## 2｜🎮 玩法說明

| 系統 | 怎麼玩 | 對應程式 |
|---|---|---|
| 💗 **笨笨的心情** | 說「請」「謝謝」、稱讚、道歉 → 心情變好、比較聽話；罵它「笨蛋」、催「快點」、連打驚嘆號、一直重複同一句 → 它會頂嘴，**鬧彆扭時甚至拒絕做事**。心情也會被事件影響（踩樂高很痛、充飽電很開心） | `gameplay/mood.py` |
| 🩸 **主人血壓** | 撞牆、踩樂高、吸到貓、闖禁區、輾過狗便便……每次出包血壓都往上飆，**爆表（200 mmHg）就失敗** | `gameplay/pressure.py` |
| 🔋 **電量** | 走一格 −1、吸一次 −2；回到充電座會充飽。**電量歸零就失敗** | `world/game.py` |
| 🎲 **突發事件** | 貓狗會亂跑、小孩打翻果汁、狗狗留下便便、門鈴響了笨笨會分心 | `gameplay/incidents.py` |
| ⭐ **分數與星級** | 連續吸乾淨有**連擊加分**；⭐ 過關、⭐⭐ 指令數 ≤ 標準、⭐⭐⭐ 再加上零違規且血壓沒超過 166 mmHg | `gameplay/scoring.py` |
| 🙃 **恍神（v3）** | 笨笨有一定機率不照做：左右不分、煞車失靈多衝一格、偷懶少走、忘記吸、做一半就累了、做完亂轉、裝沒聽到。**機率跟心情連動**（開心 ×0.5、鬧彆扭 ×2），所以好好說話真的有用 | `gameplay/quirks.py` |
| 🧠 **難度機制（v3）** | 越後面的關卡疊越多：指令上限、步數上限、一次只記得幾個動作、**導航壞掉要你指路**、笨笨只看得到附近、**停電**（畫面也只看得到附近）、會累、護欄停用 | `world/levels.py`、`session.py` |
| 🏅 **成就** | 「溫柔的主人」「哄哄就好」「吸貓大師」「摸黑掃地」「神之煞車」……共 20 個 | `gameplay/achievements.py` |
| 🔓 **關卡解鎖＋每日挑戰** | 教學關＋**10 個難度遞增的關卡**依序解鎖；「今日挑戰」每天自動產生一張新地圖（同一天大家都一樣，可以比星星） | `gameplay/progress.py`、`world/mapgen.py` |

**新障礙物**：🐶 狗（擋路、會亂跑、會大便）、💩 狗便便（輾過去會抹得到處都是）、💦 水漬（踩到滑一格，吸到會短路）。

**指令範例**：`請往前走三格然後吸`、`右轉，前進兩格`、`不准進臥室`、`繞過那隻貓`、`回去充電`、`停`、`笨笨你好棒！`
**斜線指令**：`/levels` 選關、`/level 3`、`/level daily`、`/next`、`/reset`、`/mode serious`、`/view` 換視角、`/achievements`、`/help`、`q` 離開

**看方向**（v3）：笨笨頭上的**黃色大箭頭＝正前方**；前方格子上的 **1、2、3…＝「往前走幾格」**；
藍色「左」、綠色「右」標在**笨笨自己的**左手邊、右手邊（不是螢幕的左右）；角落的指北針會把笨笨面向的方位標成黃色。
「🔄 視角」可以輪流切換 **2.5D → 俯視（上方＝北）→ 🧊 3D**。3D 預設「🎥 跟隨」相機：鏡頭跟在笨笨背後，**畫面的左右就是笨笨的左右**；「🗺️ 俯瞰」看整張地圖，拖曳轉視角、滾輪縮放（第一次切到 3D 會從網路下載 three.js）。

In [ ]:
from bendan.gameplay import Progress
from bendan.render import levels_html, show
from bendan.world import Game

display(HTML(levels_html(Progress())))
show(Game(4))      # 第 4 關「狗狗的驚喜」預覽（2.5D）
show(Game(4), view="top")   # 同一關的俯視圖

### 難度曲線

每一關都會多一個新機制；恍神率還會跟著笨笨的心情變化。

In [ ]:
from bendan.render import difficulty_html

display(HTML(difficulty_html()))

## 3｜🎮 遊戲視窗（推薦）

執行下一格就會出現遊戲畫面；**預設從你還沒破的第一關開始**。

| 想做的事 | 怎麼做 |
|---|---|
| **開成獨立的遊戲視窗** | 按右上角「🪟 彈出成獨立視窗」。關掉那個視窗（或按「⤓ 拉回 Notebook」）遊戲會回到這裡，進度不會不見 |
| 全螢幕 | 「⛶ 全螢幕」（按 Esc 離開全螢幕） |
| 下指令 | 在輸入框打字按 **Enter**（中文輸入法選字時不會誤送）；**↑／↓** 叫出之前講過的指令 |
| 緊急煞車 | 「✋ 停！」或鍵盤 **Esc** |
| 哄笨笨 | 「💗 哄一下」 |
| 看之前講過什麼 | 對話框下面的「📜 指令紀錄」：你說了什麼 → 笨笨回什麼、做了什麼、有沒有恍神 → 結果。點紀錄裡的指令可以放回輸入框 |
| 語音輸入 | **在自己電腦上**（裝了 `requirements-voice.txt`）：**按住 🎤 講話、放開送出**，用本機 Whisper 辨識；吼太大聲會被當成「大吼」。課程 JupyterHub 是 http，瀏覽器不給麥克風 → 用**電腦內建聽寫**（Mac 按兩下 fn、Windows 按 Win+H） |

> 彈出視窗被瀏覽器擋住時：點網址列右邊「已封鎖彈出式視窗」的圖示 → 一律允許，再按一次。

In [ ]:
# ▶️ 執行這格開始玩（遊戲視窗）
ui = bendan.play_widgets(mode="silly")
# 想直接玩某一關：bendan.play_widgets(level=7, unlock_all=True)

## 4｜⌨️ 文字模式（備用：任何 Jupyter 都能玩）

沒有 ipywidgets 或遊戲視窗出不來時用這個：在下方輸入框打指令，`q` 離開。對話框下面一樣有指令紀錄。

In [ ]:
# ▶️ 執行這格開始玩（文字模式）
session = bendan.play(mode="silly")

## 5｜🏅 進度與成就

In [ ]:
from bendan.render import achievements_html

progress = Progress()
display(HTML(levels_html(progress) + achievements_html(progress)))
# progress.reset()   # ← 想清空存檔重新開始，把這行的 # 拿掉再執行

## 6｜常見問題與小提醒

| 發生什麼事？ | 先檢查哪裡？ |
|---|---|
| `沒有 ipywidgets`／遊戲視窗出不來 | Kernel 選錯了：課程 JupyterHub 的「Python (Lab 1)」只有 openai，請換成 **Python 3 (ipykernel)**；或改玩第 4 節文字模式 |
| 按「彈出成獨立視窗」沒反應 | 瀏覽器擋了彈出視窗：網址列右邊的圖示 → 允許；或改用「⛶ 全螢幕」 |
| 顯示「改用離線 FakeLLM」 | `OPENAI_BASE_URL` / `OPENAI_API_KEY` 是否存在（第 0 節的檢查表會顯示） |
| `ModuleNotFoundError: bendan` | 整個專案資料夾（含 `bendan/`）有沒有一起上傳；可用 `scripts/pack_for_jupyterhub.py` 打包成 zip |
| `Model Not Found` | 1B 列出的模型名稱是否真的存在 |
| 笨笨一直「發呆」 | 02 Notebook 看模型原話；可設定 `llm.STATE.json_mode = True` |
| 笨笨一直拒絕 | 它在鬧彆扭！說「請」、稱讚或道歉；或用 `/mode serious` |
| 遊戲視窗畫面不動 | 重新執行那一格；還是不行就改用文字模式 |
| 太難了 | 前幾關多練習；說話溫柔一點（恍神率會降低）；`/mode serious` 恍神率減半 |
| 延遲太久 | 換小模型（`qwen2.5:3b`）、`play(feedback_rounds=0)` 關掉事件回饋 |

**一句話總結：** LLM 負責理解與規劃；Python 負責驗證與真的執行；遊戲世界負責回報事件；玩法系統讓「怎麼說話」也變成策略——四者只用 JSON 溝通。

## 🌟 AI 工具使用聲明（學術誠信）

> 依課程規定，請在報告最後寫清楚使用了什麼 AI 工具、如何使用。以下為範例，請依實際情況修改：

1. **使用的 AI 工具：** Claude Code（Anthropic Claude Opus 5.5）
2. **如何使用：** 依據期末專題提案 PDF 與課程 API Quickstart 範本，協助產生程式架構（等角渲染、遊戲引擎、Prompt／記憶／JSON 驗證、非同步 Session、評估程式）；
   v2 將單一 Notebook 重構為 `bendan/` 套件，並參考《教妹妹開車》等遊戲加入心情、血壓、電量、突發事件、計分、成就、每日挑戰等玩法與單元測試；
   v3 依組員試玩回饋加入 10 個難度遞增的關卡與「恍神」機制、SVG 遊戲畫面與方向標示、可彈出的遊戲視窗、指令紀錄，並評估語音輸入。
   組員再自行測試、調整人設台詞與關卡，並以課程 LLM 實際跑評估數據。